# CS 4372 Assignment 4 - Vinay A. Nair

Chosen Book: Alice's Adventures in Wonderland by Lewis Carroll

This project aims to utilize pretrained Transformer models to accomplish four NLP tasks:

1) Text Summarization
2) Machine Translation from English to French
3) Developing a question-answering system (chatbot) based on the text
4) Text generation starting from a small seed text from the book

In [ ]:
%pip install pandas==2.2.3 requests==2.32.4 bottleneck>=1.5.0

I initially attempted to obtain the book from Project Gutenberg, but the website was unavailable. As an alternative, I used Standard Ebooks to retrieve the same book from its public GitHub repository. I downloaded the repository as a ZIP file and extracted the XHTML chapter files to obtain the text.

In [ ]:
import requests
import zipfile
from pathlib import Path

url = (
    "https://github.com/standardebooks/"
    "lewis-carroll_alices-adventures-in-wonderland_john-tenniel/"
    "archive/refs/heads/master.zip"
)

response = requests.get(url, timeout=60)
response.raise_for_status()

epub_path = Path("alice_standard_ebooks_source.zip")
epub_path.write_bytes(response.content)

print("File size:", epub_path.stat().st_size)
print("Valid ZIP:", zipfile.is_zipfile(epub_path))

File size: 113720477
Valid ZIP: True


In [ ]:
import re
import textwrap
import torch
from bs4 import BeautifulSoup

chapters = {}

with zipfile.ZipFile(epub_path, "r") as archive:
    chapter_files = [
        name for name in archive.namelist()
        if re.search(r"chapter-\d+\.xhtml$", name)
    ]

    chapter_files = sorted(
        chapter_files,
        key=lambda name: int(
            re.search(r"chapter-(\d+)", name).group(1)
        )
    )

    for name in chapter_files:
        number = int(
            re.search(r"chapter-(\d+)", name).group(1)
        )

        soup = BeautifulSoup(
            archive.read(name),
            "html.parser"
        )

        paragraphs = [
            p.get_text(" ", strip=True)
            for p in soup.find_all("p")
        ]

        chapters[number] = "\n\n".join(paragraphs)

clean_text = "\n\n".join(chapters.values())
book_text = clean_text

print("Total chapters:", len(chapters))
print("Total words:", len(clean_text.split()))

print("\nFirst 500 characters:\n")
print(textwrap.fill(clean_text[:500], width=80))

Total chapters: 12
Total words: 26460

First 500 characters:

Down the Rabbit-Hole  Alice was beginning to get very tired of sitting by her
sister on the bank, and of having nothing to do: once or twice she had peeped
into the book her sister was reading, but it had no pictures or conversations in
it, “and what is the use of a book,” thought Alice “without pictures or
conversations?”  So she was considering in her own mind (as well as she could,
for the hot day made her feel very sleepy and stupid), whether the pleasure of
making a daisy-chain would be wor


I then divided the 12 chapters into 151 passages of approximately 200 words each. This allows the pretrained models to process smaller sections of the book while preserving the chapter information.

In [ ]:
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.tokenize import sent_tokenize

def split_passages(text, target_words=200):
    passages = []
    current = []
    current_words = 0

    paragraphs = text.split("\n\n")

    for paragraph in paragraphs:
        sentences = sent_tokenize(paragraph)

        for sentence in sentences:
            words = len(sentence.split())

            if current and current_words + words > target_words:
                passages.append(" ".join(current))
                current = []
                current_words = 0

            current.append(sentence)
            current_words += words

    if current:
        passages.append(" ".join(current))

    return passages

passages = []

for chapter_number, content in sorted(chapters.items()):
    chapter_passages = split_passages(content)

    for passage_number, passage in enumerate(chapter_passages, 1):
        passages.append({
            "chapter": chapter_number,
            "passage": passage_number,
            "text": passage,
            "words": len(passage.split())
        })

print("Total passages:", len(passages))
print("Average words:", round(sum(p["words"] for p in passages) / len(passages), 1))
print("\nFirst passage:\n")
print(textwrap.fill(passages[0]["text"], width=90))

Total passages: 151
Average words: 175.2

First passage:

Down the Rabbit-Hole Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into the book her sister
was reading, but it had no pictures or conversations in it, “and what is the use of a
book,” thought Alice “without pictures or conversations?” So she was considering in her
own mind (as well as she could, for the hot day made her feel very sleepy and stupid),
whether the pleasure of making a daisy-chain would be worth the trouble of getting up and
picking the daisies, when suddenly a White Rabbit with pink eyes ran close by her. There
was nothing so very remarkable in that; nor did Alice think it so very much out of the way
to hear the Rabbit say to itself, “Oh dear! Oh dear!


## 1. Text Summarization

The first task is to summarize passages from the book into 2-3 sentences that capture the main events. I chose Google's FLAN-T5-Large, a pretrained encoder-decoder Transformer model, for this task. We start by loading the model and its tokenizer.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

model.to("cpu")
model.eval()

print("Model:", model_name)
print("Parameters:", f"{model.num_parameters():,}")
print("Device:", next(model.parameters()).device)
print("Model loaded successfully")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model: google/flan-t5-large
Parameters: 783,150,080
Device: cpu
Model loaded successfully


We then create a prompt that instructs the model to summarize the passage in approximately 40-60 words. The prompt asks the model to focus on the main characters and events while avoiding information that is not present in the original text.

In [ ]:
def summarize_flan(text):
    prompt = (
        "Write a concise plot summary of this story excerpt. "
        "Describe what the main character is doing, what important event occurs, "
        "and how the situation changes. "
        "Focus on the central events rather than copying the opening sentence. "
        "Write 2 to 3 complete sentences, approximately 40 to 60 words. "
        "Preserve character names and do not invent events.\n\n"
        f"Story:\n{text}\n\nPlot summary:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to("cpu")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            min_new_tokens=45,
            max_new_tokens=100,
            num_beams=4,
            do_sample=False,
            no_repeat_ngram_size=3,
            length_penalty=1.5,
            early_stopping=True
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

sample_summary = summarize_flan(passages[0]["text"])

print("GENERATED SUMMARY:\n")
print(textwrap.fill(sample_summary, width=90))
print("\nSummary words:", len(sample_summary.split()))

GENERATED SUMMARY:

Alice was beginning to get very tired of sitting by her sister on the bank, and of having
nothing to do. Once or twice she had peeped into the book her sister was reading, but it
had no pictures or conversations in it. So she was considering in her own mind whether the
pleasure of making a daisy-chain would be worth the trouble of getting up and picking the
daisies.

Summary words: 70


The next step is to evaluate the model's performance. I selected 5 passages from different sections of the book and created reference summaries for comparison. We then generate summaries using FLAN-T5-Large and evaluate how closely they match the reference summaries.

In [ ]:
%pip install -q evaluate rouge_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
selected_indices = [0, 30, 60, 90, 120]

for i, idx in enumerate(selected_indices, 1):
    p = passages[idx]

    print("\n" + "=" * 70)
    print(f"PASSAGE {i} | CHAPTER {p['chapter']}")
    print("=" * 70 + "\n")

    for paragraph in p["text"].split("\n\n"):
        print(textwrap.fill(paragraph.strip(), width=80))
        print()

    print(f"Word count: {len(p['text'].split())}")
    print()


PASSAGE 1 | CHAPTER 1

Down the Rabbit-Hole Alice was beginning to get very tired of sitting by her
sister on the bank, and of having nothing to do: once or twice she had peeped
into the book her sister was reading, but it had no pictures or conversations in
it, “and what is the use of a book,” thought Alice “without pictures or
conversations?” So she was considering in her own mind (as well as she could,
for the hot day made her feel very sleepy and stupid), whether the pleasure of
making a daisy-chain would be worth the trouble of getting up and picking the
daisies, when suddenly a White Rabbit with pink eyes ran close by her. There was
nothing so very remarkable in that; nor did Alice think it so very much out of
the way to hear the Rabbit say to itself, “Oh dear! Oh dear!

Word count: 146


PASSAGE 2 | CHAPTER 3

“It is a long tail, certainly,” said Alice, looking down with wonder at the
Mouse’s tail; “but why do you call it sad?” And she kept on puzzling about it
while the Mouse 

In [ ]:
reference_summaries = [
    """
    Alice sits beside her sister on the bank, feeling bored
    and sleepy on a hot day. Her sister's book has neither
    pictures nor conversations, making it uninteresting to
    Alice. As she wonders whether making a daisy-chain is
    worth the effort of picking flowers, a pink-eyed White
    Rabbit runs close by. The Rabbit anxiously says that it
    will be late, but Alice does not initially find the
    talking animal particularly remarkable.
    """,

    """
    Alice hears the Mouse tell a sad tale but becomes
    distracted by its long tail, confusing the two words.
    The Mouse's story concerns Fury, who threatens to
    take another mouse to court, act as both judge and
    jury, and sentence it to death. Noticing Alice's
    inattention, the Mouse grows angry. Alice mistakes
    its protest for a mention of a knot and eagerly
    offers to help untie it, deepening their misunderstanding.
    """,

    """
    Alice unexpectedly reaches an open area where she
    finds a house only about four feet high. She realizes
    that approaching it at her current size could badly
    frighten whoever lives inside. Rather than risk
    alarming the inhabitants, she decides to make herself
    smaller before going any closer. She nibbles again at
    the right-hand piece she is holding and waits until
    she has shrunk to nine inches tall before approaching
    the little house.
    """,

    """
    Alice watches an elaborate royal procession pass by,
    beginning with club-carrying soldiers and diamond-decorated
    courtiers. Ten royal children follow, then distinguished
    guests, including a nervous White Rabbit who fails to
    notice Alice. The Knave of Hearts carries the King's
    crown on a crimson cushion, and the King and Queen
    bring up the rear. Alice wonders whether she should
    lie face down like the gardeners, but decides that
    doing so would prevent her from seeing the procession.
    """,

    """
    The Gryphon tells Alice that shoes beneath the sea
    are worked on with whiting and made from soles and
    eels, playing with the names of sea creatures. Alice
    then imagines telling a porpoise to stay away. The
    Mock Turtle insists that no sensible fish travels
    without a porpoise, turning the word into a joke
    about purpose. When Alice questions him, he takes
    offense, and the Gryphon asks to hear about her adventures.
    """
]

In [ ]:
selected_indices = [0, 30, 60, 90, 120]

flan_results = []

for idx in selected_indices:
    p = passages[idx]
    summary = summarize_flan(p["text"])

    flan_results.append({
        "chapter": p["chapter"],
        "passage": p["passage"],
        "original": p["text"],
        "summary": summary,
        "original_words": len(p["text"].split()),
        "summary_words": len(summary.split())
    })

for i, result in enumerate(flan_results, 1):
    print("\n" + "=" * 70)
    print(f"SUMMARY {i} | CHAPTER {result['chapter']}")
    print("=" * 70 + "\n")

    print(textwrap.fill(result["summary"], width=80))

    print(f"\nOriginal words: {result['original_words']}")
    print(f"Summary words: {result['summary_words']}")


SUMMARY 1 | CHAPTER 1

Alice was beginning to get very tired of sitting by her sister on the bank, and
of having nothing to do. Once or twice she had peeped into the book her sister
was reading, but it had no pictures or conversations in it. So she was
considering in her own mind whether the pleasure of making a daisy-chain would
be worth the trouble of getting up and picking the daisies.

Original words: 146
Summary words: 70

SUMMARY 2 | CHAPTER 3

Alice and Fury meet in the house. Fury wants to prosecute the Mouse. The Mouse
refuses to go to the trial. Alice helps the Mouse undo a knot in the Mouse's
tail. They go to court.

Original words: 194
Summary words: 36

SUMMARY 3 | CHAPTER 5

Alice is a mouse. She is hungry and wants to get into the garden. As she nibbles
at the righthand bit again, she comes upon an open place, with a little house in
it about four feet high.

Original words: 97
Summary words: 38

SUMMARY 4 | CHAPTER 8

The White Rabbit is talking in a hurried nervous man

We now compare the generated summaries with the reference summaries using ROUGE scores. ROUGE measures the overlap between the generated and reference texts, allowing us to evaluate how well the model preserves important information.

In [ ]:
import evaluate
rouge = evaluate.load("rouge")

all_scores = []

for i in range(5):
    scores = rouge.compute(
        predictions=[flan_results[i]["summary"]],
        references=[reference_summaries[i]]
    )

    all_scores.append(scores)

    print(f"\nPASSAGE {i + 1}")
    print("-" * 30)

    for metric, score in scores.items():
        print(f"{metric}: {score:.4f}")

print("\nAVERAGE ROUGE SCORES")
print("=" * 30)

for metric in all_scores[0]:
    average = sum(
        scores[metric] for scores in all_scores
    ) / len(all_scores)

    print(f"{metric}: {average:.4f}")


PASSAGE 1
------------------------------
rouge1: 0.4167
rouge2: 0.1268
rougeL: 0.3194
rougeLsum: 0.3611

PASSAGE 2
------------------------------
rouge1: 0.3853
rouge2: 0.1121
rougeL: 0.2385
rougeLsum: 0.3486

PASSAGE 3
------------------------------
rouge1: 0.3964
rouge2: 0.1468
rougeL: 0.1982
rougeLsum: 0.3784

PASSAGE 4
------------------------------
rouge1: 0.3220
rouge2: 0.0345
rougeL: 0.1356
rougeLsum: 0.2542

PASSAGE 5
------------------------------
rouge1: 0.2913
rouge2: 0.0792
rougeL: 0.2136
rougeLsum: 0.2913

AVERAGE ROUGE SCORES
rouge1: 0.3623
rouge2: 0.0999
rougeL: 0.2211
rougeLsum: 0.3267


We then calculate and store the average ROUGE-1, ROUGE-2, ROUGE-L, and ROUGE-Lsum scores across the 5 evaluated passages.

In [ ]:
flan_evaluation = {
    "model": "google/flan-t5-large",
    "num_passages": len(all_scores),
    "rouge1": sum(s["rouge1"] for s in all_scores) / len(all_scores),
    "rouge2": sum(s["rouge2"] for s in all_scores) / len(all_scores),
    "rougeL": sum(s["rougeL"] for s in all_scores) / len(all_scores),
    "rougeLsum": sum(s["rougeLsum"] for s in all_scores) / len(all_scores)
}

print("FLAN-T5 summarization evaluation complete")
print(flan_evaluation)

FLAN-T5 summarization evaluation complete
{'model': 'google/flan-t5-large', 'num_passages': 5, 'rouge1': np.float64(0.36233603964158184), 'rouge2': np.float64(0.09987795326580381), 'rougeL': np.float64(0.22107204121661553), 'rougeLsum': np.float64(0.3267225533516844)}


## 2. Machine Translation to French

The second task is to translate English passages from the book into French. I chose Helsinki-NLP's opus-mt-en-fr, a pretrained encoder-decoder Transformer model designed for English-to-French translation. We start by loading the model and its tokenizer before translating the selected passages.

In [ ]:
%pip install -q sentencepiece sacremoses

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cpu


In [ ]:
translation_model_name = "Helsinki-NLP/opus-mt-en-fr"

translation_tokenizer = AutoTokenizer.from_pretrained(
    translation_model_name
)

translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    translation_model_name
).to(device)

translation_model.eval()

print("Model:", translation_model_name)
print("Device:", device)
print("Parameters:", f"{translation_model.num_parameters():,}")

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  301MB            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model: Helsinki-NLP/opus-mt-en-fr
Device: cpu
Parameters: 75,133,952


In [ ]:
english_passage = passages[0]["text"]

inputs = translation_tokenizer(
    english_passage,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

with torch.inference_mode():
    outputs = translation_model.generate(
        **inputs,
        max_new_tokens=400,
        num_beams=2
    )

french_translation = translation_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("\n" + "=" * 70)
print("ORIGINAL ENGLISH | CHAPTER 1")
print("=" * 70 + "\n")

print(textwrap.fill(english_passage, width=80))

print("\n" + "=" * 70)
print("FRENCH TRANSLATION | HELSINKI-NLP")
print("=" * 70 + "\n")

print(textwrap.fill(french_translation, width=80))

print("\n" + "=" * 70)
print("WORD COUNTS")
print("=" * 70)

print("English words:", len(english_passage.split()))
print("French words:", len(french_translation.split()))

[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ORIGINAL ENGLISH | CHAPTER 1

Down the Rabbit-Hole Alice was beginning to get very tired of sitting by her
sister on the bank, and of having nothing to do: once or twice she had peeped
into the book her sister was reading, but it had no pictures or conversations in
it, “and what is the use of a book,” thought Alice “without pictures or
conversations?” So she was considering in her own mind (as well as she could,
for the hot day made her feel very sleepy and stupid), whether the pleasure of
making a daisy-chain would be worth the trouble of getting up and picking the
daisies, when suddenly a White Rabbit with pink eyes ran close by her. There was
nothing so very remarkable in that; nor did Alice think it so very much out of
the way to hear the Rabbit say to itself, “Oh dear! Oh dear!

FRENCH TRANSLATION | HELSINKI-NLP

Au fond du Lapin-Hole Alice commençait à se lasser de s'asseoir près de sa sœur
sur la banque, et de n'avoir rien à faire: une ou deux fois elle avait piétiné
dans le li

In [ ]:
selected_indices = [0, 30, 60, 90, 120]

translation_results = []

for idx in selected_indices:
    p = passages[idx]

    inputs = translation_tokenizer(
        p["text"],
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.inference_mode():
        outputs = translation_model.generate(
            **inputs,
            max_new_tokens=400,
            num_beams=2
        )

    french = translation_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    translation_results.append({
        "chapter": p["chapter"],
        "original": p["text"],
        "translation": french
    })

print("Translations generated:", len(translation_results))

[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Translations generated: 5


In [ ]:
selected_indices = [0, 30, 60, 90, 120]

for i, idx in enumerate(selected_indices, 1):
    p = passages[idx]

    print("\n" + "=" * 70)
    print(f"PASSAGE {i} | CHAPTER {p['chapter']}")
    print("=" * 70 + "\n")

    for paragraph in p["text"].split("\n\n"):
        print(textwrap.fill(paragraph.strip(), width=80))
        print()

    print(f"Word count: {len(p['text'].split())}")
    print()


PASSAGE 1 | CHAPTER 1

Down the Rabbit-Hole Alice was beginning to get very tired of sitting by her
sister on the bank, and of having nothing to do: once or twice she had peeped
into the book her sister was reading, but it had no pictures or conversations in
it, “and what is the use of a book,” thought Alice “without pictures or
conversations?” So she was considering in her own mind (as well as she could,
for the hot day made her feel very sleepy and stupid), whether the pleasure of
making a daisy-chain would be worth the trouble of getting up and picking the
daisies, when suddenly a White Rabbit with pink eyes ran close by her. There was
nothing so very remarkable in that; nor did Alice think it so very much out of
the way to hear the Rabbit say to itself, “Oh dear! Oh dear!

Word count: 146


PASSAGE 2 | CHAPTER 3

“It is a long tail, certainly,” said Alice, looking down with wonder at the
Mouse’s tail; “but why do you call it sad?” And she kept on puzzling about it
while the Mouse 

To evaluate the translation model, I used Google Translate to create French reference translations for the 5 selected English passages. These references are then compared with the translations generated by Helsinki-NLP.

In [ ]:
french_passages = [
    """
    Dans le terrier du lapin. Alice commençait à se lasser
    terriblement de rester assise sur la berge, près de sa sœur,
    sans rien avoir à faire. À une ou deux reprises, elle avait
    jeté un coup d’œil au livre que lisait sa sœur, mais il ne
    contenait ni images ni dialogues ; et à quoi peut bien
    servir un livre, se dit Alice, s’il n’y a ni images ni
    dialogues ? Elle se demandait donc — du mieux qu’elle
    pouvait, car la chaleur de la journée la rendait très
    somnolente et engourdie — si cela valait la peine de se
    lever pour cueillir des pâquerettes et en faire une guirlande,
    quand, soudain, un Lapin blanc aux yeux roses passa tout
    près d’elle. Il n’y avait là rien de bien extraordinaire ;
    et Alice ne trouva pas non plus très surprenant d’entendre
    le Lapin se dire à lui-même : Oh là là ! Oh là là !
    Je vais être en retard !
    """,

    """
    C’est une longue queue, assurément, dit Alice en regardant
    avec étonnement la queue de la Souris ; mais pourquoi
    dites-vous qu’elle est triste ? Et, tandis que la Souris
    parlait, elle continuait à se creuser la tête à ce sujet,
    si bien qu’elle finit par se représenter l’histoire à peu
    près comme suit : Furie dit à une souris, qu’il rencontra
    dans la maison : Allons en justice tous les deux ; je vais
    te poursuivre en justice. Allons, ne discute pas ; il nous
    faut un procès, car, en vérité, je n’ai rien à faire ce matin.
    La souris répondit au chien : Un tel procès, cher Monsieur,
    sans jury ni juge, ne serait que perte de temps.
    Je serai le juge, je serai le jury, dit le rusé vieux
    Furie ; je mènerai tout le procès et je te condamnerai
    à mort. Vous ne m’écoutez pas ! dit sévèrement
    la Souris à Alice. À quoi pensez-vous ?
    Je vous demande pardon, dit Alice très humblement ;
    vous en étiez arrivée à la cinquième boucle, je crois ?
    Pas du tout ! s’écria la Souris, sèchement et très
    en colère. Un nœud ! dit Alice, toujours prête à se
    rendre utile, tout en regardant autour d’elle avec
    inquiétude. Oh, laissez-moi vous aider à le défaire !
    """,

    """
    Tout en disant cela, elle arriva soudain dans un espace
    dégagé, où se trouvait une petite maison d’environ quatre
    pieds de haut. Qui que ce soit qui habite là, pensa Alice,
    il ne faudrait surtout pas que je me présente devant eux
    avec cette taille : je leur ferais une peur terrible !
    Elle recommença donc à grignoter le morceau de droite
    et n’osa pas s’approcher de la maison avant d’avoir
    réduit sa taille à neuf pouces de haut.
    """,

    """
    D’abord vinrent dix soldats portant des massues ; ils
    avaient tous la même forme que les trois jardiniers,
    longs et plats, avec les mains et les pieds aux coins.
    Ensuite vinrent dix courtisans, ornés de carreaux sur
    tout le corps, qui marchaient deux par deux, comme
    les soldats. Après eux arrivèrent les enfants royaux ;
    ils étaient dix, et ces chers petits avançaient
    joyeusement en sautillant, main dans la main, par couples.
    Ils étaient tous décorés de cœurs. Puis vinrent les
    invités, principalement des Rois et des Reines, parmi
    lesquels Alice reconnut le Lapin blanc. Il parlait
    d’un air pressé et nerveux, souriait à tout ce qu’on
    disait et passa sans la remarquer. Ensuite venait
    le Valet de Cœur, portant la couronne du Roi sur un
    coussin de velours cramoisi ; et enfin, pour terminer
    cette grande procession, arrivèrent le Roi et la Reine
    de Cœur. Alice se demanda si elle ne devait pas se
    coucher face contre terre comme les trois jardiniers,
    mais elle ne se souvenait pas avoir jamais entendu
    parler d’une telle règle pour les processions.
    Et d’ailleurs, pensa-t-elle, à quoi servirait une
    procession si tout le monde devait se coucher sur
    le visage, de sorte que personne ne puisse la voir ?
    """,

    """
    Les bottes et les chaussures sous la mer, continua
    le Griffon d’une voix grave, sont travaillées avec
    un merlan. Maintenant, tu le sais.
    Et de quoi sont-elles faites ? demanda Alice
    avec beaucoup de curiosité.
    De soles et d’anguilles, bien sûr, répondit
    le Griffon avec une certaine impatience.
    N’importe quelle crevette aurait pu te le dire.
    Si j’avais été le merlan, dit Alice, dont les pensées
    étaient encore occupées par la chanson, j’aurais dit
    au marsouin : Restez à l’écart, s’il vous plaît :
    nous ne voulons pas de vous avec nous !
    Ils étaient obligés de l’emmener avec eux,
    dit la Fausse Tortue. Aucun poisson sensé ne
    voyagerait sans marsouin.
    Vraiment ? demanda Alice, très surprise.
    Bien sûr que non, répondit la Fausse Tortue.
    Si un poisson venait me voir et me disait qu’il
    partait en voyage, je lui demanderais :
    Avec quel marsouin ?
    Ne voulez-vous pas dire dans quel but ?
    demanda Alice.
    Je dis exactement ce que je veux dire,
    répondit la Fausse Tortue d’un ton offensé.
    Et le Griffon ajouta :
    Allons, racontons maintenant quelques-unes
    de tes aventures.
    """
]

In [ ]:
%pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sacrebleu]


We then evaluate the model using SacreBLEU to calculate BLEU and chrF scores. BLEU measures the overlap of word sequences between the generated and reference translations, while chrF measures character-level similarity.

In [ ]:
import sacrebleu
import pandas as pd

generated_translations = [
    result["translation"] for result in translation_results
]

references = [" ".join(p.split()) for p in french_passages]
predictions = [" ".join(p.split()) for p in generated_translations]

assert len(predictions) == len(references) == 5

results = []

for i, (prediction, reference) in enumerate(zip(predictions, references)):
    bleu = sacrebleu.sentence_bleu(prediction, [reference]).score
    chrf = sacrebleu.sentence_chrf(prediction, [reference]).score

    results.append({
        "Passage": i + 1,
        "Chapter": translation_results[i]["chapter"],
        "BLEU": bleu,
        "chrF": chrf
    })

df = pd.DataFrame(results)

display(df.round(2))

print("Corpus BLEU:", round(
    sacrebleu.corpus_bleu(predictions, [references]).score, 2
))

print("Corpus chrF:", round(
    sacrebleu.corpus_chrf(predictions, [references]).score, 2
))

,Passage,Chapter,BLEU,chrF
0,1,1,18.99,47.09
1,2,3,0.78,10.03
2,3,5,10.73,45.75
3,4,8,16.53,44.70
4,5,10,1.77,21.96


Corpus BLEU: 10.72
Corpus chrF: 32.24


We then store the corpus BLEU and chrF scores to summarize the machine translation model's performance across the 5 selected passages.

In [ ]:
translation_evaluation = {
    "model": "Helsinki-NLP/opus-mt-en-fr",
    "num_passages": len(predictions),
    "corpus_bleu": sacrebleu.corpus_bleu(
        predictions, [references]
    ).score,
    "corpus_chrf": sacrebleu.corpus_chrf(
        predictions, [references]
    ).score
}

print("Machine translation evaluation complete")
print(translation_evaluation)

Machine translation evaluation complete
{'model': 'Helsinki-NLP/opus-mt-en-fr', 'num_passages': 5, 'corpus_bleu': 10.721426604575083, 'corpus_chrf': 32.24223184520999}


## 3. Developing a Question-Answering System (Chatbot)

The third task is to develop a question-answering system based on the book. I chose DistilBERT (distilbert-base-cased-distilled-squad), a pretrained encoder-only Transformer model designed for extractive question answering. The model identifies answers directly from the provided text.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

qa_model_name = "distilbert/distilbert-base-cased-distilled-squad"

qa_tokenizer = AutoTokenizer.from_pretrained(
    qa_model_name
)

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    qa_model_name
).to(device)

qa_model.eval()

print("QA model loaded successfully")

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

QA model loaded successfully


We first test the model using the first passage and a simple question: "What animal ran close by Alice?" The expected answer is the White Rabbit, which the model correctly identifies from the passage.

In [ ]:
question = "What animal ran close by Alice?"
context = passages[0]["text"]

inputs = qa_tokenizer(
    question,
    context,
    return_tensors="pt",
    truncation="only_second",
    max_length=512
).to(device)

with torch.inference_mode():
    outputs = qa_model(**inputs)

start = outputs.start_logits.argmax().item()
end = outputs.end_logits.argmax().item()

answer = qa_tokenizer.decode(
    inputs["input_ids"][0][start:end + 1],
    skip_special_tokens=True
)

print("QUESTION:\n")
print(question)

print("\nANSWER:\n")
print(answer)

QUESTION:

What animal ran close by Alice?

ANSWER:

White Rabbit


We then index all 151 passages using TF-IDF so that the question-answering system can retrieve relevant information from throughout the book. Cosine similarity is used to identify the passages most closely related to a given question.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(
    stop_words="english"
)

passage_vectors = vectorizer.fit_transform(
    [p["text"] for p in passages]
)

def retrieve_passages(question, k=3):
    query_vector = vectorizer.transform([question])

    scores = cosine_similarity(
        query_vector,
        passage_vectors
    )[0]

    indices = scores.argsort()[::-1][:k]

    return [
        {
            "chapter": passages[i]["chapter"],
            "text": passages[i]["text"],
            "score": scores[i]
        }
        for i in indices
    ]

print("Indexed passages:", passage_vectors.shape[0])

Indexed passages: 151


We test the retrieval system using the same question and examine the most relevant passages along with their similarity scores. This helps verify that the system retrieves appropriate context before generating an answer.

In [ ]:
question = "What animal ran close by Alice?"

results = retrieve_passages(question)

for i, result in enumerate(results, 1):
    print("\n" + "=" * 60)
    print(f"RESULT {i} | CHAPTER {result['chapter']}")
    print(f"Similarity: {result['score']:.4f}")
    print("=" * 60 + "\n")

    print(textwrap.fill(result["text"][:400], width=80))


RESULT 1 | CHAPTER 4
Similarity: 0.1305

Heads below!” (a loud crash)⁠—“Now, who did that?⁠—It was Bill, I fancy⁠—Who’s
to go down the chimney?⁠—Nay, I shan’t! You do it!⁠—That I won’t, then!⁠—Bill’s
to go down⁠—Here, Bill! the master says you’re to go down the chimney!” “Oh! So
Bill’s got to come down the chimney, has he?” said Alice to herself. “Shy, they
seem to put everything upon Bill! I wouldn’t be in Bill’s place for a good deal:

RESULT 2 | CHAPTER 1
Similarity: 0.1168

Down the Rabbit-Hole Alice was beginning to get very tired of sitting by her
sister on the bank, and of having nothing to do: once or twice she had peeped
into the book her sister was reading, but it had no pictures or conversations in
it, “and what is the use of a book,” thought Alice “without pictures or
conversations?” So she was considering in her own mind (as well as she could,
for the hot da

RESULT 3 | CHAPTER 12
Similarity: 0.1009

At this the whole pack rose up into the air, and came flying down upon h

We then create a helper function that combines passage retrieval with the DistilBERT question-answering model. The function retrieves the 3 most relevant passages, extracts candidate answers, and returns the highest-scoring answer along with its source chapter.

In [ ]:
def ask_book(question, k=3):
    retrieved = retrieve_passages(question, k)

    best_answer = ""
    best_score = float("-inf")
    best_chapter = None

    for passage in retrieved:
        inputs = qa_tokenizer(
            question,
            passage["text"],
            return_tensors="pt",
            truncation="only_second",
            max_length=512
        ).to(device)

        with torch.inference_mode():
            outputs = qa_model(**inputs)

        start = outputs.start_logits.argmax().item()
        end = outputs.end_logits.argmax().item()

        if end < start:
            continue

        answer = qa_tokenizer.decode(
            inputs["input_ids"][0][start:end + 1],
            skip_special_tokens=True
        )

        score = (
            outputs.start_logits[0][start]
            + outputs.end_logits[0][end]
        ).item()

        if score > best_score:
            best_answer = answer
            best_score = score
            best_chapter = passage["chapter"]

    return {
        "question": question,
        "answer": best_answer,
        "chapter": best_chapter
    }

We now test the complete question-answering function to verify that it returns the expected answer and identifies the chapter from which the answer was extracted.

In [ ]:
question = "What animal ran close by Alice?"

result = ask_book(question)

print("QUESTION:\n")
print(result["question"])

print("\nANSWER:\n")
print(result["answer"])

print("\nSOURCE CHAPTER:", result["chapter"])

QUESTION:

What animal ran close by Alice?

ANSWER:

White Rabbit

SOURCE CHAPTER: 1


To evaluate the question-answering system, I created 5 questions based on events and characters from different chapters of the book. Each question has an expected answer that we use to assess the model's accuracy.

In [ ]:
qa_test_cases = [
    {
        "question": "What animal ran close by Alice?",
        "expected": "White Rabbit"
    },
    {
        "question": "Who threatened to prosecute the Mouse?",
        "expected": "Fury"
    },
    {
        "question": "How tall did Alice become before approaching the little house?",
        "expected": "nine inches"
    },
    {
        "question": "Who carried the King's crown?",
        "expected": "Knave of Hearts"
    },
    {
        "question": "Who asked Alice to tell some of her adventures?",
        "expected": "Gryphon"
    }
]

In [ ]:
qa_results = []

for i, test in enumerate(qa_test_cases, 1):
    result = ask_book(test["question"])

    predicted = result["answer"]
    expected = test["expected"]

    qa_results.append({
        "question": test["question"],
        "expected": expected,
        "predicted": predicted,
        "chapter": result["chapter"]
    })

    print("\n" + "=" * 70)
    print(f"QUESTION {i}")
    print("=" * 70)

    print("\nQuestion:", test["question"])
    print("Expected:", expected)
    print("Predicted:", predicted)
    print("Chapter:", result["chapter"])


QUESTION 1

Question: What animal ran close by Alice?
Expected: White Rabbit
Predicted: White Rabbit
Chapter: 1

QUESTION 2

Question: Who threatened to prosecute the Mouse?
Expected: Fury
Predicted: Fury
Chapter: 3

QUESTION 3

Question: How tall did Alice become before approaching the little house?
Expected: nine inches
Predicted: four feet high
Chapter: 5

QUESTION 4

Question: Who carried the King's crown?
Expected: Knave of Hearts
Predicted: Knave of Hearts
Chapter: 8

QUESTION 5

Question: Who asked Alice to tell some of her adventures?
Expected: Gryphon
Predicted: Gryphon
Chapter: 10


We then compare the model's predicted answers with the expected answers using Exact Match and F1 Score. Exact Match measures whether the normalized answers are identical, while F1 measures the overlap between the words in the predicted and expected answers.

In [ ]:
from collections import Counter
def normalize_answer(text):
    text = text.lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return " ".join(text.split())

def exact_match(predicted, expected):
    return int(
        normalize_answer(predicted) == normalize_answer(expected)
    )

def f1_score(predicted, expected):
    pred_tokens = normalize_answer(predicted).split()
    gold_tokens = normalize_answer(expected).split()

    common = Counter(pred_tokens) & Counter(gold_tokens)
    overlap = sum(common.values())

    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

em_scores = []
f1_scores = []

for i, result in enumerate(qa_results, 1):
    em = exact_match(result["predicted"], result["expected"])
    f1 = f1_score(result["predicted"], result["expected"])

    em_scores.append(em)
    f1_scores.append(f1)

    print(f"Question {i}: EM = {em}, F1 = {f1:.4f}")

print("\nAVERAGE EXACT MATCH:", sum(em_scores) / len(em_scores))
print("AVERAGE F1:", sum(f1_scores) / len(f1_scores))

Question 1: EM = 1, F1 = 1.0000
Question 2: EM = 1, F1 = 1.0000
Question 3: EM = 0, F1 = 0.0000
Question 4: EM = 1, F1 = 1.0000
Question 5: EM = 1, F1 = 1.0000

AVERAGE EXACT MATCH: 0.8
AVERAGE F1: 0.8


In [ ]:
qa_evaluation = {
    "model": "distilbert/distilbert-base-cased-distilled-squad",
    "retrieval": "TF-IDF",
    "num_questions": 5,
    "exact_match": sum(em_scores) / len(em_scores),
    "f1": sum(f1_scores) / len(f1_scores)
}

print("Question-answering evaluation complete")

Question-answering evaluation complete


## 4. Text Generation Starting from a Small Seed Text

The fourth task is to generate new text using a short passage from the book as the starting point. I chose DistilGPT2, a pretrained decoder-only Transformer model that generates text by predicting one token at a time based on the preceding context.

We first experiment with different seed lengths and generation settings before producing continuations for 5 selected passages. The model is evaluated using perplexity on the original text.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

generation_model_name = "distilbert/distilgpt2"

generation_tokenizer = AutoTokenizer.from_pretrained(
    generation_model_name
)

generation_model = AutoModelForCausalLM.from_pretrained(
    generation_model_name
).to(device)

generation_model.eval()

print("Model:", generation_model_name)
print("Device:", device)
print("Parameters:", generation_model.num_parameters())

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model: distilbert/distilgpt2
Device: cpu
Parameters: 81912576


In [ ]:
seed_text = " ".join(
    passages[0]["text"].split()[:20]
)

print("SEED TEXT:\n")
print(textwrap.fill(seed_text, width=80))

SEED TEXT:

Down the Rabbit-Hole Alice was beginning to get very tired of sitting by her
sister on the bank, and of


In [ ]:
inputs = generation_tokenizer(
    seed_text,
    return_tensors="pt"
).to(device)

with torch.inference_mode():
    outputs = generation_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.15,
        pad_token_id=generation_tokenizer.eos_token_id
    )

generated_text = generation_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("\n" + "=" * 70)
print("SEED TEXT")
print("=" * 70 + "\n")

print(textwrap.fill(seed_text, width=80))

print("\n" + "=" * 70)
print("GENERATED STORY")
print("=" * 70 + "\n")

print(textwrap.fill(generated_text, width=80))

print("\nGenerated words:", len(generated_text.split()))


SEED TEXT

Down the Rabbit-Hole Alice was beginning to get very tired of sitting by her
sister on the bank, and of

GENERATED STORY

Down the Rabbit-Hole Alice was beginning to get very tired of sitting by her
sister on the bank, and of course we decided that it would be nice to have a
little bit more time with them when she could. So after hearing about this all
over again I thought: What do you think?

Generated words: 54


In [ ]:
seed_text = " ".join(
    passages[0]["text"].split()[:60]
)

inputs = generation_tokenizer(
    seed_text,
    return_tensors="pt"
).to(device)

with torch.inference_mode():
    outputs = generation_model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.85,
        top_k=40,
        repetition_penalty=1.1,
        pad_token_id=generation_tokenizer.eos_token_id
    )

generated_text = generation_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("\n" + "=" * 70)
print("SEED TEXT")
print("=" * 70 + "\n")

print(textwrap.fill(seed_text, width=80))

print("\n" + "=" * 70)
print("GENERATED CONTINUATION")
print("=" * 70 + "\n")

continuation = generated_text[len(seed_text):].strip()

print(textwrap.fill(continuation, width=80))


SEED TEXT

Down the Rabbit-Hole Alice was beginning to get very tired of sitting by her
sister on the bank, and of having nothing to do: once or twice she had peeped
into the book her sister was reading, but it had no pictures or conversations in
it, “and what is the use of a book,” thought Alice “without pictures or
conversations?”

GENERATED CONTINUATION

The first time I read this came when my grandmother told me that there were four
books for Christmas. The two stories are not about Harry Potter; they're about
his life after he left school as an adult (a boy) who goes out with him only
because of Hogwarts' rules against "behave" – which has always been quite
unwisely interpreted since its inception. It's all just one story — you have to
choose your own path from home to work until at least ten years later, where
Hermione wants to live before going back down to see if things can be better
together so


In [ ]:
selected_indices = [0, 30, 60, 90, 120]

generation_results = []

for idx in selected_indices:
    p = passages[idx]

    seed = " ".join(p["text"].split()[:60])

    inputs = generation_tokenizer(
        seed,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():
        outputs = generation_model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.7,
            top_p=0.85,
            top_k=40,
            repetition_penalty=1.1,
            pad_token_id=generation_tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]

    continuation = generation_tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    generation_results.append({
        "chapter": p["chapter"],
        "seed": seed,
        "continuation": continuation
    })

print("Generated continuations:", len(generation_results))

Generated continuations: 5


In [ ]:
for i, result in enumerate(generation_results, 1):
    print("\n" + "=" * 70)
    print(f"GENERATION {i} | CHAPTER {result['chapter']}")
    print("=" * 70)

    print("\nSEED TEXT:\n")
    print(textwrap.fill(result["seed"], width=80))

    print("\nGENERATED CONTINUATION:\n")
    print(textwrap.fill(result["continuation"], width=80))

    print("\nGenerated words:", len(result["continuation"].split()))


GENERATION 1 | CHAPTER 1

SEED TEXT:

Down the Rabbit-Hole Alice was beginning to get very tired of sitting by her
sister on the bank, and of having nothing to do: once or twice she had peeped
into the book her sister was reading, but it had no pictures or conversations in
it, “and what is the use of a book,” thought Alice “without pictures or
conversations?”

GENERATED CONTINUATION:

 The rabbit looked up at me. She glanced over my shoulder for an answer. I
didn't know how many books there were; so far only one appeared here when we
first met. My brother would be right next to us (it's hard to imagine that he'd
ever read this), while his friends could easily tell you all about him as if
they knew everything else from history. He did not even bother talking with
anyone except myself who wasn�t really interested in anything other than
literature—he just loved stories because he felt like someone wanted to see
something new happen every time he saw them

Generated words: 105

GENERATION

In [ ]:
import math

total_loss = 0
total_tokens = 0
perplexity_results = []

for idx in selected_indices:
    p = passages[idx]
    evaluation_text = " ".join(p["text"].split()[60:])

    if not evaluation_text.strip():
        continue

    inputs = generation_tokenizer(
        evaluation_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.inference_mode():
        outputs = generation_model(
            **inputs,
            labels=inputs["input_ids"]
        )

    token_count = inputs["input_ids"].shape[1] - 1

    if token_count <= 0:
        continue

    loss = outputs.loss.item()
    perplexity = math.exp(loss)

    total_loss += loss * token_count
    total_tokens += token_count

    perplexity_results.append({
        "chapter": p["chapter"],
        "perplexity": perplexity
    })

    print(
        f"Chapter {p['chapter']}: "
        f"Perplexity = {perplexity:.2f}"
    )

corpus_perplexity = math.exp(total_loss / total_tokens)

print("\nCORPUS PERPLEXITY:", round(corpus_perplexity, 2))

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Chapter 1: Perplexity = 57.32
Chapter 3: Perplexity = 37.27
Chapter 5: Perplexity = 56.51
Chapter 8: Perplexity = 102.30
Chapter 10: Perplexity = 13.99

CORPUS PERPLEXITY: 34.2


In [ ]:
generation_evaluation = {
    "model": "distilbert/distilgpt2",
    "num_passages": 5,
    "corpus_perplexity": corpus_perplexity,
    "passage_perplexities": perplexity_results
}

print("Text generation evaluation complete")

Text generation evaluation complete


# Final Evaluation Results

We now summarize the evaluation results obtained from all four NLP tasks:

1) Text Summarization - ROUGE Scores
2) Machine Translation - BLEU and chrF Scores
3) Question Answering - Exact Match and F1 Score
4) Text Generation - Perplexity

These metrics help us assess the performance of the pretrained Transformer models on the selected passages from the book.

In [ ]:
print("=" * 70)
print("TASK 1: SUMMARIZATION")
print("=" * 70)
print(flan_evaluation)

print("\n" + "=" * 70)
print("TASK 2: MACHINE TRANSLATION")
print("=" * 70)
print(translation_evaluation)

print("\n" + "=" * 70)
print("TASK 3: QUESTION ANSWERING")
print("=" * 70)
print(qa_evaluation)

print("\n" + "=" * 70)
print("TASK 4: TEXT GENERATION")
print("=" * 70)
print(generation_evaluation)

TASK 1: SUMMARIZATION
{'model': 'google/flan-t5-large', 'num_passages': 5, 'rouge1': np.float64(0.36233603964158184), 'rouge2': np.float64(0.09987795326580381), 'rougeL': np.float64(0.22107204121661553), 'rougeLsum': np.float64(0.3267225533516844)}

TASK 2: MACHINE TRANSLATION
{'model': 'Helsinki-NLP/opus-mt-en-fr', 'num_passages': 5, 'corpus_bleu': 10.721426604575083, 'corpus_chrf': 32.24223184520999}

TASK 3: QUESTION ANSWERING
{'model': 'distilbert/distilbert-base-cased-distilled-squad', 'retrieval': 'TF-IDF', 'num_questions': 5, 'exact_match': 0.8, 'f1': 0.8}

TASK 4: TEXT GENERATION
{'model': 'distilbert/distilgpt2', 'num_passages': 5, 'corpus_perplexity': 34.19841344588919, 'passage_perplexities': [{'chapter': 1, 'perplexity': 57.316894802163276}, {'chapter': 3, 'perplexity': 37.26771290522827}, {'chapter': 5, 'perplexity': 56.51398904038924}, {'chapter': 8, 'perplexity': 102.29912066499169}, {'chapter': 10, 'perplexity': 13.985452111101667}]}
